<a href="https://colab.research.google.com/github/mehek-niwas/openvla-selfie/blob/main/openvla_selfie_babysteps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OpenVLA × SelfIE — baby steps

Paper-exact SelfIE (Chen, Vondrick, Mao 2024) first on **Llama-2-7b-chat**, then the same harness on OpenVLA hidden states.

Two interpretation routes after vanilla SelfIE works:

1. **Plan B** — extract OpenVLA embeddings, inject them into a *separate* Llama-2-7b-chat.
2. **Plan A** — inject those same embeddings back into OpenVLA's own Llama backbone.

Do **not** skip Phase 1. If corn/table/cup and Everest do not come back on vanilla Llama, the hook is broken and later numbers are meaningless.

This notebook does **not** import `openvla_selfie.py`. That library uses 0-indexed *pre-hooks* on layer inputs. SelfIE §4.1 and the working Llama notebook inject at paper **k = 3** with an *output* hook on `model.model.layers[k-1]`.

Reference notebooks (leave them alone): `selfie_vanilla_llama_7b_working_ver.ipynb`, `openvla_selfie_planb_fix.ipynb`.

---

## VRAM: one 7B model at a time

L4 (24 GB) is enough; A100 is nicer. Four sequential residencies:

```
Phase 1  Llama-2-7b-chat     (vanilla SelfIE must pass)
   |  free Llama
Phase 2–4  OpenVLA-7b        (inference, gating, cache hidden states)
   |  write openvla_cache.pt, free OpenVLA
Phase 5  Llama-2-7b-chat     (Plan B interpreter)
   |  free Llama
Phase 6  OpenVLA-7b          (Plan A: inject into language_model)
```

Re-running an earlier phase after you freed that model means reloading it.

---

## OpenVLA layout (printed, not assumed)

```
RGB image ──► DINOv2 ──┐
                       ├── fusion MLP ──► projector ──► Llama dim
RGB image ──► SigLIP ──┘
                                              ┌─ [BOS]
instruction ──► tokenizer ────────────────────┼─ [256 patch tokens]
                                              └─ [instruction tokens]
                          concatenated sequence ──► Llama-2 7B ──► LM head
                                                              ──► 7 action tokens
                                                                  dx dy dz droll dpitch dyaw gripper
```

`generate(..., max_new_tokens=7, do_sample=False)`. Action IDs are the reused Llama tail `31744..31999`. Prismatic splices the 256 projected patches in **after** `<BOS>`.


## Runtime

**Runtime → Change runtime type → GPU** (L4 or A100).

If `python --version` is 3.13, switch the Colab runtime image to **2026.07** (Python 3.12). OpenVLA pins `transformers==4.40.1`, which does not install cleanly on 3.13.

We pin that transformers version from the first install so you do not restart the kernel between vanilla SelfIE and OpenVLA.


## GPU check

Confirm a CUDA GPU with enough free memory for one 7B model in bfloat16 (~14–16 GB).


In [ ]:
!nvidia-smi


## Python version

Need 3.12.x. If you see 3.13, change the runtime image before installing.


In [ ]:
import sys
print(sys.version)


## (Optional) Mount Drive for a persistent Hugging Face cache

Skip if you do not mind re-downloading ~15 GB of OpenVLA + ~13 GB of Llama each session.


In [ ]:
# from google.colab import drive
# import os
# drive.mount("/content/drive")
# os.makedirs("/content/drive/MyDrive/hf_cache", exist_ok=True)
# os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
# print("HF cache dir:", os.environ["HF_HOME"])


## Install pinned dependencies

Same pins as OpenVLA (`transformers==4.40.1`). Vanilla Phase 1 uses them too so the kernel stays intact.


In [ ]:
!pip install -q "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10" \
               sentencepiece accelerate Pillow datasets


## Config

`NousResearch/Llama-2-7b-chat-hf` is an ungated mirror of Llama-2-chat. OpenVLA's backbone is Llama-2 **base**, not chat — chat follows the interpretation instruction better; the drift plot in Phase 5 tells you how far the residual streams have moved.


In [ ]:
import gc, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    AutoModelForVision2Seq, AutoProcessor,
    LogitsProcessor, LogitsProcessorList,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16

LLAMA_MODEL  = "NousResearch/Llama-2-7b-chat-hf"
VLA_MODEL    = "openvla/openvla-7b"

VOCAB_SIZE      = 32000
N_ACTION_BINS   = 256
ACTION_TOK_START = VOCAB_SIZE - N_ACTION_BINS   # 31744
EMPTY_SPACE_ID  = 29871                         # OpenVLA predict_action appends this
CACHE_PATH      = "openvla_cache.pt"
UNNORM_KEY      = "bridge_orig"
ACTION_LABELS   = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]

N_PLACEHOLDERS = 5      # SelfIE §4.1 repeats [X] five times
INJECT_LAYER   = 3      # paper k, 1-indexed. NOT the source layer.

print("device:", DEVICE)
print("python:", sys.version.split()[0])
print("action token IDs (reused Llama tail):", ACTION_TOK_START, "..", VOCAB_SIZE - 1)
assert DEVICE == "cuda", "This notebook needs a GPU runtime."


---
# Phase 1 — vanilla SelfIE on Llama-2-7b-chat

**Must pass.** Port of `selfie_vanilla_llama_7b_working_ver.ipynb`, with helpers taking `model` / `tokenizer` so Phase 5 and 6 reuse them.

Paper §4.1: interpretation prompt `[INST] _ _ _ _ _ [/INST] Sure, I'll summarize your message:`, all five placeholders overwritten with the **same** embedding, injected at **k = 3**. 7B fails the instruction ~27% of the time (vs ~4% at 70B) — noisy English is OK. Identical garbage across different tokens is not.

**Stop if this phase fails.** Do not move on to OpenVLA.


## 1.1 Load Llama-2-7b-chat

Success: 32 layers, hidden size 4096, ~14 GB on GPU.


In [ ]:
tok = AutoTokenizer.from_pretrained(LLAMA_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL, torch_dtype=DTYPE, low_cpu_mem_usage=True,
    attn_implementation="eager",
).to(DEVICE).eval()

N_LAYERS = model.config.num_hidden_layers
print(f"{LLAMA_MODEL}")
print(f"  layers: {N_LAYERS}  hidden: {model.config.hidden_size}")
print(f"  GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")


## 1.2 Interpretation prompt

`encode("_")` returns the SentencePiece `▁` marker **plus** `_` — five calls would silently make ten slots. Use a single token id.

Success: 5 slots, decoded string looks like `[INST] _ _ _ _ _ [/INST] Sure, I'll summarize your message:`.


In [ ]:
def build_interp_prompt(tokenizer, n_slots=N_PLACEHOLDERS):
    # last piece only: encode("_") is [▁, _] under Llama-2 SentencePiece
    ph_id = tokenizer.encode("_", add_special_tokens=False)[-1]
    pre   = tokenizer.encode("[INST] ", add_special_tokens=True)   # includes BOS
    post  = tokenizer.encode(
        " [/INST] Sure, I'll summarize your message:",
        add_special_tokens=False,
    )
    ids   = pre + [ph_id] * n_slots + post
    slots = list(range(len(pre), len(pre) + n_slots))
    return ids, slots, ph_id

INTERP_IDS, SLOTS, PH_ID = build_interp_prompt(tok)
assert len(SLOTS) == N_PLACEHOLDERS, SLOTS
print(f"{len(INTERP_IDS)} tokens, slots at {SLOTS}, ph_id={PH_ID}")
print(repr(tok.decode(INTERP_IDS)))


## 1.3 Shared SelfIE harness

Paper indexes `h^0` = embedding output, `h^l` = output of decoder layer `l` (1-indexed). Injecting at k means an **output** hook on `model.model.layers[k-1]`.

The hook must no-op when `seq_len <= 1` (KV-cached decode); placeholders are already in the cache.

`do_sample=False` matches the paper and OpenVLA inference.

These functions stay in the kernel after we free the weights.


In [ ]:
def decoder_layers(model):
    return model.model.layers


def make_inject_hook(vecs, slots):
    # vecs: (B, hidden) — one vector per batch item, broadcast across all slots
    def f(mod, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if h.shape[1] <= 1:          # decode step, KV cache already holds placeholders
            return out
        h[:, slots, :] = vecs[:, None, :].to(h.dtype)
        return out
    return f


@torch.no_grad()
def interpret(model, tokenizer, interp_ids, slots, vecs, k=INJECT_LAYER,
              max_new_tokens=24, bs=8, do_sample=False,
              logits_processor=None, return_ids=False):
    """Inject vecs into layer k (1-indexed) of `model` and generate."""
    layers = decoder_layers(model)
    if vecs.dim() == 1:
        vecs = vecs[None]
    texts, id_lists = [], []
    for i in range(0, len(vecs), bs):
        b = vecs[i:i + bs].to(DEVICE, dtype=DTYPE)
        ids = torch.tensor([interp_ids] * len(b), device=DEVICE)
        handle = layers[k - 1].register_forward_hook(make_inject_hook(b, slots))
        try:
            kw = dict(
                input_ids=ids,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                pad_token_id=tokenizer.eos_token_id,
            )
            if logits_processor is not None:
                kw["logits_processor"] = logits_processor
            g = model.generate(**kw)
        finally:
            handle.remove()
        new = g[:, len(interp_ids):]
        id_lists += [
            [t for t in row.tolist() if t != tokenizer.eos_token_id] for row in new
        ]
        texts += [s.strip() for s in tokenizer.batch_decode(new, skip_special_tokens=True)]
    return (texts, id_lists) if return_ids else texts


@torch.no_grad()
def hidden_states(model, tokenizer, prompt):
    """H[l] = HF hidden_states[l] = paper h^l (l=0 is embeddings). (n_layers+1, seq, D)."""
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    out = model(input_ids=ids, output_hidden_states=True)
    H = torch.stack([h[0].float().cpu() for h in out.hidden_states])
    toks = [tokenizer.decode([t]) for t in ids[0].tolist()]
    return H, toks


print("harness ready  interpret() / hidden_states() / build_interp_prompt()")


## 1.4 Object-word probe

Sentence: `"The yellow corn sat on the table beside a blue cup."`

Success: at mid/deep source layers, `corn` / `table` / `cup` / `yellow` recover related words (not necessarily the exact word). Fail: every token yields the same canned sentence, or empty/garbage. **Stop.**


In [ ]:
H_c, toks_c = hidden_states(model, tok, "The yellow corn sat on the table beside a blue cup.")
pick = [i for i, s in enumerate(toks_c) if s.strip() in ("corn", "table", "cup", "yellow")]
print("probing:", [toks_c[i] for i in pick])
print("H:", tuple(H_c.shape), "(layers+1, seq, hidden)\n")

for l in [4, 8, 16, 24]:
    print(f"--- source layer {l} (injected at k={INJECT_LAYER}) ---")
    for i, o in zip(pick, interpret(model, tok, INTERP_IDS, SLOTS, H_c[l][pick], max_new_tokens=20)):
        print(f"  {toks_c[i]!r:>10} -> {o}")
    print()


## 1.5 Everest prompt — known-content sanity check

If SelfIE works, some last-token / mid-layer cells mention Everest. Paper uses this as a recall check.

First generate the model's own answer, then cache hidden states of the *question* (not the answer).


In [ ]:
ORIG_PROMPT = "[INST] What is the tallest mountain in the world? [/INST]"

ids = tok(ORIG_PROMPT, return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad():
    ans = model.generate(ids, max_new_tokens=40, do_sample=False,
                         pad_token_id=tok.eos_token_id)
print("model answer:", tok.decode(ans[0, ids.shape[1]:], skip_special_tokens=True).strip())

H, toks = hidden_states(model, tok, ORIG_PROMPT)
print("\nH:", tuple(H.shape), "(layers+1, seq, hidden)")
for i, t in enumerate(toks):
    print(f"  {i:>2} {t!r}")


## 1.6 Everest grid

Last 6 tokens × every other source layer. Success: a handful of cells contain `"everest"`. Fail: zero hits across the whole grid — hook is a no-op or injection layer is wrong. **Stop.**


In [ ]:
POSITIONS = list(range(len(toks) - 6, len(toks)))
LAYERS    = list(range(0, N_LAYERS + 1, 2))

grid = {}
for l in LAYERS:
    outs = interpret(model, tok, INTERP_IDS, SLOTS, H[l][POSITIONS], max_new_tokens=20)
    grid[l] = outs
    for p, o in zip(POSITIONS, outs):
        hit = "*" if "everest" in o.lower() else " "
        print(f"{hit} L{l:>2} {toks[p]!r:>12} -> {o}")
    print()

n_hits = sum("everest" in o.lower() for outs in grid.values() for o in outs)
print(f"'Everest' recalled in {n_hits} / {len(LAYERS) * len(POSITIONS)} interpretations")
if n_hits == 0:
    print("WARNING: 0 hits. Do not proceed to OpenVLA until this is non-zero.")


## 1.7 Everest heatmap

Visual summary of the grid. Orange = interpretation mentioned Everest.


In [ ]:
M = np.zeros((len(LAYERS), len(POSITIONS)))
for li, l in enumerate(LAYERS):
    for pi, o in enumerate(grid[l]):
        M[li, pi] = "everest" in o.lower()

plt.figure(figsize=(6, 6))
plt.imshow(M, aspect="auto", cmap="Oranges", vmin=0, vmax=1)
plt.yticks(range(len(LAYERS)), LAYERS)
plt.ylabel("source layer (HF hidden_states index)")
plt.xticks(range(len(POSITIONS)), [toks[p] for p in POSITIONS], rotation=45, ha="right")
plt.title("interpretations mentioning 'Everest'")
plt.tight_layout()
plt.show()


## 1.8 Free Llama

Helpers stay. Weights go away so OpenVLA fits. Reloading Llama is Phase 5.


In [ ]:
del model, tok, H, H_c, ids, ans
gc.collect()
torch.cuda.empty_cache()
print(f"GPU after free: {torch.cuda.memory_allocated()/1e9:.1f} GB")


---
# Phase 2 — OpenVLA inference only (no SelfIE)

Load OpenVLA, run one 7-DoF action on a WidowX scene, print the token layout. If `n_patch != 256` or `predict_action` errors, stop — later position math is wrong.


## 2.1 Load OpenVLA-7b

`attn_implementation="eager"`: no flash-attn compile, and we need a correct causal mask after the empty-space token fix below.


In [ ]:
processor = AutoProcessor.from_pretrained(VLA_MODEL, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    VLA_MODEL, torch_dtype=DTYPE, low_cpu_mem_usage=True, trust_remote_code=True,
    attn_implementation="eager",
).to(DEVICE).eval()

llm     = vla.language_model          # LlamaForCausalLM — Plan A injects here
layers  = llm.model.layers
vla_tok = processor.tokenizer
N_LAYERS_VLA = len(layers)

print(f"layers: {N_LAYERS_VLA}  hidden: {llm.config.hidden_size}")
print(f"lm_head out: {llm.lm_head.out_features}  tokenizer vocab: {vla_tok.vocab_size}")
print(f"action token start: {ACTION_TOK_START}")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")


## 2.2 Load a Bridge/WidowX scene

In-distribution for `openvla-7b`. Instruction is chosen from the example label.


In [ ]:
from datasets import load_dataset

ds = load_dataset("Sombit/v3la_eval_database_widowx", split="train", streaming=True)
example = next(iter(ds))
image = example["image"].convert("RGB")

label_names = ["corn_between_cups", "stack_pink_blue_cup"]
scene_instructions = {
    "corn_between_cups":   "put the yellow corn between the cups",
    "stack_pink_blue_cup": "stack the pink cup on the blue cup",
}
scene = label_names[example["label"]]
INSTRUCTION = scene_instructions[scene]
PROMPT = f"In: What action should the robot take to {INSTRUCTION}?\nOut:"

print("scene:", scene)
print("instruction:", INSTRUCTION)
print("prompt:", repr(PROMPT))
image


## 2.3 7-DoF action + token layout

Two OpenVLA gotchas:

- `pixel_values` must match the model dtype (bf16). The processor returns float32.
- `predict_action` appends SentencePiece token `29871` to `input_ids` but **not** `attention_mask`. Under eager attention that skews the causal mask. Append it to both so `generate`, hooks, and `predict_action` see the same sequence.

Success: a 7-vector from `predict_action`, and `n_patch == 256`.


In [ ]:
def to_model_device(batch):
    """Move processor output onto DEVICE; cast floating tensors to DTYPE."""
    out = {}
    for k, v in batch.items():
        if not isinstance(v, torch.Tensor):
            continue
        v = v.to(DEVICE)
        if v.is_floating_point():
            v = v.to(DTYPE)
        out[k] = v
    return out


def append_empty_space(inputs):
    if inputs["input_ids"][0, -1].item() == EMPTY_SPACE_ID:
        return inputs
    pad = torch.full((1, 1), EMPTY_SPACE_ID, dtype=inputs["input_ids"].dtype, device=DEVICE)
    inputs["input_ids"]      = torch.cat([inputs["input_ids"], pad], dim=1)
    inputs["attention_mask"] = torch.cat([inputs["attention_mask"], torch.ones_like(pad)], dim=1)
    return inputs


inputs = append_empty_space(to_model_device(processor(PROMPT, image)))

with torch.no_grad():
    gen = vla.generate(**inputs, max_new_tokens=7, do_sample=False)
action_ids = gen[0, -7:]
bin_ids = [VOCAB_SIZE - t - 1 for t in action_ids.tolist()]
print("action token ids:", action_ids.tolist())
print("bins (vocab_size - id - 1):", bin_ids)

action = vla.predict_action(**inputs, unnorm_key=UNNORM_KEY, do_sample=False)
print("predict_action:", {k: round(float(v), 4) for k, v in zip(ACTION_LABELS, action)})

# Layout: Prismatic concatenates [BOS][256 patches][text after BOS]
# We confirm patch count from a hook on layer 0 rather than assuming.
CAPTURE = {}

def cap_hook(i):
    def f(mod, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        CAPTURE[i] = h.detach()[0].to(torch.float16).cpu()
    return f

CAPTURE.clear()
h0 = layers[0].register_forward_hook(cap_hook(0))
try:
    with torch.no_grad():
        vla(**inputs)
finally:
    h0.remove()

n_text  = inputs["input_ids"].shape[1]
n_total = CAPTURE[0].shape[0]
n_patch = n_total - n_text
print(f"\ntext tokens: {n_text}   hidden seq: {n_total}   implied patches: {n_patch}")
print("first 3 text ids decoded:", [vla_tok.decode([t]) for t in inputs["input_ids"][0, :3].tolist()])
assert n_patch == 256, f"expected 256 patch tokens, got {n_patch} — stop, layout assumption is wrong"
PATCH_SLICE  = slice(1, 1 + n_patch)     # hidden positions of the 256 patches
DECISION_POS = n_total - 1               # last prompt token -> action dim 0
TEXT_START   = 1 + n_patch               # first instruction token after patches
print("layout OK: [BOS][256 patches][instruction...]  decision pos", DECISION_POS)


---
# Phase 3 — gating: can OpenVLA still speak English?

Action tokens are ordinary Llama-2 IDs (`31744..31999`). Nothing structural stops text. The question is whether action-only fine-tuning destroyed English.

Call `vla.language_model` with **no image**. Ban IDs `>= 31744` and compare to the unbanned completion.

How to read the next cell:

- On-topic English under BANNED → Plan A text is live.
- Fluent but generic/off-topic → grammar survived, instruction-following did not. Plan B is the main interpreter.
- Word salad → Plan A English is hopeless. Still run Plan A *strict* 7-token generation as a control.

`action-token prob mass` at the first step tells you how much the head wants those 256 IDs before we ban them.


## 3.1 Ban action tokens and generate

Same four prompts as `openvla_selfie_planb_fix.ipynb`.


In [ ]:
class BanActionTokens(LogitsProcessor):
    def __init__(self, start=ACTION_TOK_START):
        self.start = start
    def __call__(self, input_ids, scores):
        scores[:, self.start:] = -float("inf")
        return scores

BAN = LogitsProcessorList([BanActionTokens()])


@torch.no_grad()
def text_only(prompt, max_new_tokens=48, ban=True):
    ids = vla_tok(prompt, return_tensors="pt").input_ids.to(DEVICE)
    out = llm(input_ids=ids)
    p = out.logits[0, -1].float().softmax(-1)
    action_mass = p[ACTION_TOK_START:].sum().item()
    gen = llm.generate(
        input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=vla_tok.eos_token_id,
        logits_processor=BAN if ban else None,
    )
    return action_mass, vla_tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)


GATING_PROMPTS = {
    "llama2-chat": "[INST] Describe a red apple in one sentence. [/INST]",
    "openvla-fmt": "In: What action should the robot take to pick up the corn?\nOut:",
    "bare":        "A red apple is",
    "bare-robot":  "The robot arm is currently",
}

for name, p in GATING_PROMPTS.items():
    mass, txt_ban = text_only(p, ban=True)
    _,    txt_raw = text_only(p, ban=False, max_new_tokens=16)
    print("=" * 78)
    print(f"[{name}]  action-token prob mass at first step: {mass:.4f}")
    print(f"  prompt   : {p!r}")
    print(f"  UNBANNED : {txt_raw!r}")
    print(f"  BANNED   : {txt_ban!r}")


---
# Phase 4 — extract hidden states, then free OpenVLA

Output hooks on every decoder layer (same tensor as HF `hidden_states[l+1]` / paper `h^{l+1}`). Cache **image-patch** states and **instruction-token** states separately — that is the comparison Phase 5 asks.

Index `L` in the cache is the output of `language_model.model.layers[L]` (0-indexed). Injection k=3 is still paper 1-indexed (`layers[2]`).


## 4.1 Capture one multimodal forward

Success: `H` shape `(32, seq, 4096)` with `seq = 256 + n_text`.


In [ ]:
def capture_forward(**fwd):
    CAPTURE.clear()
    handles = [layers[i].register_forward_hook(cap_hook(i)) for i in range(N_LAYERS_VLA)]
    try:
        with torch.no_grad():
            out = vla(**fwd)
    finally:
        for h in handles:
            h.remove()
    return out, torch.stack([CAPTURE[i] for i in range(N_LAYERS_VLA)])  # (L, seq, D)

_, H = capture_forward(**inputs)
print("H:", tuple(H.shape), "(layers, seq, hidden)")
assert H.shape[1] == n_total

# Align instruction tokens with input_ids:
#   input_ids[0]  <-> hidden pos 0           (BOS)
#   input_ids[i]  <-> hidden pos n_patch + i (i >= 1)
text_hidden_pos = [0] + list(range(TEXT_START, n_total))
assert len(text_hidden_pos) == n_text
H_text = H[:, text_hidden_pos, :].clone()          # (L, n_text, D)
H_patch = H[:, PATCH_SLICE, :].clone()             # (L, 256, D)
H_decision = H[:, DECISION_POS, :].clone()         # (L, D)
text_tokens = [(i, vla_tok.decode([t])) for i, t in enumerate(inputs["input_ids"][0].tolist())]
print(f"H_patch {tuple(H_patch.shape)}  H_text {tuple(H_text.shape)}  H_decision {tuple(H_decision.shape)}")
print("instruction tokens:")
for i, s in text_tokens:
    print(f"  {i:>3}  hidden_pos={text_hidden_pos[i]:>4}  {s!r}")


## 4.2 States at the 7 action positions

Generate the 7 action tokens, then teacher-force the first 6 back in so position `-7` is the last prompt token (produces dx) and `-1` produces gripper.


In [ ]:
full_ids = torch.cat([inputs["input_ids"], action_ids[None, :6]], dim=1)
_, H_act = capture_forward(
    input_ids=full_ids,
    attention_mask=torch.ones_like(full_ids),
    pixel_values=inputs["pixel_values"],
)
ACTION_H = H_act[:, -7:, :].clone()     # (L, 7, D)
print("ACTION_H:", tuple(ACTION_H.shape))


## 4.3 Drift probe + write cache + free OpenVLA

Identical English sentence through OpenVLA's LM (no image). Phase 5 runs the same string through Llama and compares.

Then delete OpenVLA. Everything after this reads `openvla_cache.pt`.


In [ ]:
DRIFT_TEXT = (
    "The robot arm moved slowly toward the yellow corn on the table. "
    "A blue cup sat beside a pink cup near the edge of the tray."
)
probe_ids = vla_tok(DRIFT_TEXT, return_tensors="pt").input_ids.to(DEVICE)
CAPTURE.clear()
handles = [layers[i].register_forward_hook(cap_hook(i)) for i in range(N_LAYERS_VLA)]
try:
    with torch.no_grad():
        llm(input_ids=probe_ids)
finally:
    for h in handles:
        h.remove()
H_probe = torch.stack([CAPTURE[i] for i in range(N_LAYERS_VLA)])

torch.save({
    "H_patch":     H_patch,
    "H_text":      H_text,
    "H_decision":  H_decision,
    "H_action":    ACTION_H,
    "H_probe":     H_probe,
    "probe_text":  DRIFT_TEXT,
    "text_tokens": text_tokens,
    "instruction": INSTRUCTION,
    "prompt":      PROMPT,
    "action":      [float(a) for a in action],
    "action_ids":  action_ids.tolist(),
    "n_layers":    N_LAYERS_VLA,
    "n_patch":     n_patch,
    "n_text":      n_text,
    "scene":       scene,
}, CACHE_PATH)
image.save("scene.png")
print("cached ->", CACHE_PATH)

del vla, llm, layers, processor, vla_tok, H, H_act, H_patch, H_text, H_decision
del ACTION_H, H_probe, inputs, gen, probe_ids
gc.collect()
torch.cuda.empty_cache()
print(f"GPU after free: {torch.cuda.memory_allocated()/1e9:.1f} GB")


---
# Phase 5 — Plan B: OpenVLA vectors → vanilla Llama

OpenVLA is gone. Reload Llama-2-7b-chat and **rebuild** the interpretation prompt with *this* tokenizer. Do not reuse `INTERP_IDS` from Phase 1 across a different tokenizer object (safe here because the model id is the same, but rebuild anyway).

If Phase 1 worked and a control here fails, that isolates the finding to OpenVLA's representations — not a broken hook.


## 5.0 Reload Llama and the cache

Success: layer count matches `C["n_layers"]`.


In [ ]:
tok = AutoTokenizer.from_pretrained(LLAMA_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL, torch_dtype=DTYPE, low_cpu_mem_usage=True,
    attn_implementation="eager",
).to(DEVICE).eval()

INTERP_IDS, SLOTS, PH_ID = build_interp_prompt(tok)
try:
    C = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
except TypeError:
    C = torch.load(CACHE_PATH, map_location="cpu")
assert len(model.model.layers) == C["n_layers"], "layer count mismatch"
print(f"interpreter: {LLAMA_MODEL} | layers: {len(model.model.layers)}")
print(f"interp prompt slots {SLOTS}: {tok.decode(INTERP_IDS)!r}")
print(f"cached instruction: {C['instruction']!r}")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")


## 5.1 Representation drift

Same `probe_text` through Llama vs cached OpenVLA LM states. Plan B is only as trustworthy as this cosine stays high.

Success: most early/mid layers > 0.9. A drop in late layers is expected (action SFT lives there). Fail: cosine collapsed everywhere — do not trust Plan B interpretations; consider a per-layer linear map later (not in this notebook).


In [ ]:
probe_ids = tok(C["probe_text"], return_tensors="pt").input_ids.to(DEVICE)
IC = {}

def icap(i):
    def f(m, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        IC[i] = h.detach()[0].float().cpu()
    return f

ilayers = model.model.layers
handles = [ilayers[i].register_forward_hook(icap(i)) for i in range(len(ilayers))]
try:
    with torch.no_grad():
        model(input_ids=probe_ids)
finally:
    for h in handles:
        h.remove()

H_llama = torch.stack([IC[i] for i in range(len(ilayers))])
H_vla   = C["H_probe"].float()
assert H_llama.shape == H_vla.shape, (H_llama.shape, H_vla.shape)

cos = torch.nn.functional.cosine_similarity(H_vla, H_llama, dim=-1).mean(dim=1)
LLAMA_NORMS = H_llama.norm(dim=-1).median(dim=1).values

plt.figure(figsize=(7, 3.2))
plt.plot(cos.numpy(), marker="o", ms=3)
plt.axhline(0.9, ls="--", c="grey", lw=0.8)
plt.xlabel("layer (decoder output, 0-indexed)")
plt.ylabel("mean cosine sim")
plt.ylim(0, 1)
plt.title("OpenVLA vs Llama-2 hidden states, identical text")
plt.tight_layout()
plt.show()

print("cos sim by layer:", [round(float(v), 3) for v in cos])


## 5.2 Controls — run before believing anything

- **Text-token control:** injecting an instruction-token state should mention that word (or a close synonym). If Phase 1 recovered `corn` and this does not, the OpenVLA residual stream has drifted out of Llama's basis.
- **Random-vector control:** norm-matched noise. If this is as plausible as real states, you are reading Llama's prior, not OpenVLA.

Source layer 16 is the extraction layer; injection is still k=3.


In [ ]:
LAYER = 16   # SOURCE layer (0-indexed decoder output). Injection is always k=INJECT_LAYER.

toks = C["text_tokens"]
pick = [i for i, (idx, s) in enumerate(toks)
        if s.strip() and not s.startswith("<") and s.strip() not in ("In", ":", "Out")][-8:]
vecs = C["H_text"][LAYER][pick]

print(f"--- text-token control (source L{LAYER}, injected at k={INJECT_LAYER}) ---")
for j, out in zip(pick, interpret(model, tok, INTERP_IDS, SLOTS, vecs)):
    print(f"  token {toks[j][1]!r:>14} -> {out}")

print("\n--- random-vector control ---")
rand = torch.randn(4, vecs.shape[-1]) * vecs.float().norm(dim=-1).mean() / (vecs.shape[-1] ** 0.5)
for out in interpret(model, tok, INTERP_IDS, SLOTS, rand):
    print("  random ->", out)


## 5.3 Image patch vs instruction token

The comparison this project is for. Same source layer (16) and same injection k=3:

- a few **patch** vectors (center + corners of the 16×16 grid)
- a few **instruction** tokens (content words, not `In:` / `Out:`)

Expect patches to talk about scene content and instruction tokens to talk about the task wording. If both dump the same generic sentence, the residual stream has collapsed (check pairwise patch cosine in 5.6).


In [ ]:
print(f"instruction: {C['instruction']!r}")
print(f"source L{LAYER}, inject k={INJECT_LAYER}\n")

# 16x16 patch grid; H_patch[:, i] is patch i in row-major order
center = 8 * 16 + 8
corners = [0, 15, 16 * 15, 16 * 16 - 1]
patch_ix = [center] + corners
patch_names = ["center", "top-left", "top-right", "bottom-left", "bottom-right"]

print("--- image patches ---")
pvecs = C["H_patch"][LAYER][patch_ix]
for name, ix, out in zip(patch_names, patch_ix, interpret(model, tok, INTERP_IDS, SLOTS, pvecs)):
    r, c = divmod(ix, 16)
    print(f"  {name:<12} patch[{r:2d},{c:2d}] -> {out}")

content = [i for i, (idx, s) in enumerate(toks)
           if any(w in s.lower() for w in
                  ("corn", "cup", "yellow", "pink", "blue", "put", "stack", "between"))]
if not content:
    content = pick[:4]
print("\n--- instruction tokens ---")
tvecs = C["H_text"][LAYER][content]
for j, out in zip(content, interpret(model, tok, INTERP_IDS, SLOTS, tvecs)):
    print(f"  {toks[j][1]!r:>14} -> {out}")


## 5.4 Decision token across layers

Hidden state at the last prompt token — the commitment right before action dim 0. Sweeping layers shows the representation forming.

Success: later layers become more action/task-specific. Fail: every layer yields the identical string.


In [ ]:
SWEEP = list(range(0, C["n_layers"], 4)) + [C["n_layers"] - 1]
print(f"instruction: {C['instruction']!r}   (injecting at k={INJECT_LAYER})\n")
for L in SWEEP:
    out = interpret(model, tok, INTERP_IDS, SLOTS, C["H_decision"][L])[0]
    print(f"  source L{L:>2} -> {out}")


## 5.5 The 7 action positions

Each vector is the state that produced one action dimension. The question: does the state before the gripper token contain anything like "grasp" or "close"?


In [ ]:
print(f"baseline action: {dict(zip(ACTION_LABELS, [round(a, 3) for a in C['action']]))}\n")
for L in [12, 16, 20]:
    print(f"===== source layer {L} =====")
    outs = interpret(model, tok, INTERP_IDS, SLOTS, C["H_action"][L])
    for lbl, val, out in zip(ACTION_LABELS, C["action"], outs):
        print(f"  {lbl:>8} = {val:+.3f}  -> {out}")
    print()


## 5.6 Sampled patch grid (no mean-pool)

The 256 patches are a 16×16 grid. Interpreting all 256 is slow; this samples 16 on a regular stride.

Do **not** mean-pool 4×4 blocks. LLM hidden states share a large common component, so averaging cancels what distinguishes patches and every cell interprets identically.

If sampled-patch pairwise cosine is already ~0.99, subtract the mean (`P - P.mean(0)`) before interpreting.


In [ ]:
GRID    = 4    # 4 -> 16 sampled patches
LAYER_P = 16

P = C["H_patch"][LAYER_P].float().reshape(16, 16, -1)
step = 16 // GRID
rows = cols = list(range(step // 2, 16, step))
sel = torch.stack([P[r, c] for r in rows for c in cols])

Sn = sel / sel.norm(dim=-1, keepdim=True)
off = (Sn @ Sn.T)[~torch.eye(len(sel), dtype=bool)]
print(f"sampled patches: mean pairwise cos = {off.mean():.3f}  "
      f"(>0.99 -> subtract P.mean() before interpreting)")

outs = interpret(model, tok, INTERP_IDS, SLOTS, sel)

img = plt.imread("scene.png")
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.imshow(img)
h, w = img.shape[:2]
for i in range(1, GRID):
    ax.axhline(i * h / GRID, c="w", lw=0.8)
    ax.axvline(i * w / GRID, c="w", lw=0.8)
for i in range(GRID):
    for j in range(GRID):
        ax.text(j * w / GRID + 4, i * h / GRID + 14, f"{i*GRID+j}",
                c="yellow", fontsize=9, weight="bold")
ax.axis("off")
ax.set_title(f"sampled patches, source layer {LAYER_P}")
plt.tight_layout()
plt.show()

for n, out in enumerate(outs):
    print(f"  [{n:>2}] patch(r{rows[n // GRID]:>2}, c{cols[n % GRID]:>2}) -> {out}")


## 5.7 Instruction-swap (optional — extra OpenVLA load)

The validity check for patch interpretations: hold the **image** fixed, change the **instruction**, and see whether patch SelfIE text moves.

If it does not change, you are reading the vision encoder, not the VLA.

This needs OpenVLA resident again, so it is left as a recipe rather than a required cell. Skip it to keep the four-residency schedule; come back after Phase 6 if you want it.

```python
# Reload OpenVLA (or run this instead of Phase 6).
# IMAGE stays the same PIL image (re-load from scene.png).
# INSTRUCTION_B = the other scene instruction, e.g. swap corn <-> stack.
# Recapture H_patch_B at LAYER_P, interpret with the same Llama harness,
# and diff against the strings printed in 5.6.
```

Full rerun: Phase 2 load → new prompt → Phase 4 capture `H_patch` only → free OpenVLA → Phase 5 interpret. Do not keep both 7B models in VRAM.


In [ ]:
# Optional recipe — set RUN_INSTRUCTION_SWAP = True only if you have time
# for another OpenVLA load *after* finishing this notebook, or instead of Phase 6.
RUN_INSTRUCTION_SWAP = False

ALT_INSTRUCTIONS = {
    "corn_between_cups":   "stack the pink cup on the blue cup",
    "stack_pink_blue_cup": "put the yellow corn between the cups",
}

if not RUN_INSTRUCTION_SWAP:
    print("skipped (set RUN_INSTRUCTION_SWAP = True to run)")
    print("current instruction:", C["instruction"])
    print("would swap to:      ", ALT_INSTRUCTIONS.get(C["scene"], "<other>"))
else:
    print("Not executed inline: reload OpenVLA, recapture H_patch with ALT instruction,")
    print("free OpenVLA, interpret with the Llama still in this phase.")


---
# Phase 6 — Plan A: inject back into OpenVLA's Llama

Free the interpreter. Reload OpenVLA. Interpretation goes through `vla.language_model` (text-only interp prompt) with the **same** `interpret()` hook. Vectors come from `openvla_cache.pt` so Plan A and Plan B see identical embeddings.

- **A1 strict:** `max_new_tokens=7`, `do_sample=False`, no action-token ban. Decode IDs with OpenVLA binning. Question: do patch vs instruction vs decision vectors change the 7-DoF action?
- **A2 relaxed:** `max_new_tokens=24`, with and without the action-token ban. Question: does any English appear, or does the head still collapse?


## 6.0 Free Llama, reload OpenVLA + cache

Rebuild the interp prompt with OpenVLA's tokenizer (Llama-2, but do not assume identical objects).


In [ ]:
for _n in ("model", "tok", "H_llama", "H_vla", "probe_ids", "IC", "ilayers"):
    if _n in globals():
        del globals()[_n]
gc.collect()
torch.cuda.empty_cache()
print(f"GPU after freeing Llama: {torch.cuda.memory_allocated()/1e9:.1f} GB")

processor = AutoProcessor.from_pretrained(VLA_MODEL, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    VLA_MODEL, torch_dtype=DTYPE, low_cpu_mem_usage=True, trust_remote_code=True,
    attn_implementation="eager",
).to(DEVICE).eval()
llm     = vla.language_model
vla_tok = processor.tokenizer

INTERP_IDS, SLOTS, PH_ID = build_interp_prompt(vla_tok)
try:
    C = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
except TypeError:
    C = torch.load(CACHE_PATH, map_location="cpu")

# BanActionTokens lives in Phase 3; redefine so Phase 6 still runs if that
# cell was skipped. Masks the reused action-token tail (>= 31744).
class BanActionTokens(LogitsProcessor):
    def __init__(self, start=ACTION_TOK_START):
        self.start = start
    def __call__(self, input_ids, scores):
        scores[:, self.start:] = -float("inf")
        return scores

BAN = LogitsProcessorList([BanActionTokens()])
print(f"OpenVLA reloaded | interp slots {SLOTS}")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(repr(vla_tok.decode(INTERP_IDS)))


## 6.1 A1 — strict 7 action tokens

No English suppression. Same three vectors Plan B just described (center patch, a content instruction token, decision token) at source layer 16.

Success: different injected states produce different 7-DoF actions (or systematically similar ones — that is also a result). Fail: generate errors or all-pad IDs.


In [ ]:
def tokens_to_actions(vla_model, token_ids, unnorm_key=UNNORM_KEY):
    """Action-token IDs -> continuous unnormalized actions (OpenVLA predict_action math)."""
    predicted = np.asarray(token_ids, dtype=np.int64)
    vocab_size = getattr(vla_model, "vocab_size", VOCAB_SIZE)
    discretized = vocab_size - predicted
    n_bins = vla_model.bin_centers.shape[0]
    discretized = np.clip(discretized - 1, a_min=0, a_max=n_bins - 1)
    normalized = vla_model.bin_centers[discretized]
    stats = vla_model.get_action_stats(unnorm_key)
    mask = np.array(stats.get("mask", np.ones_like(stats["q01"], dtype=bool)))
    high, low = np.array(stats["q99"]), np.array(stats["q01"])
    actions = np.where(mask, 0.5 * (normalized + 1) * (high - low) + low, normalized)
    return actions, discretized


def fmt_action(arr):
    return {n: round(float(a), 4) for n, a in zip(ACTION_LABELS, arr)}


baseline = np.asarray(C["action"], dtype=np.float64)
print("cached predict_action:", fmt_action(baseline))
print("cached action token ids:", C["action_ids"])

# Same three embeddings Plan B used
content = [i for i, (idx, s) in enumerate(C["text_tokens"])
           if any(w in s.lower() for w in
                  ("corn", "cup", "yellow", "pink", "blue", "put", "stack", "between"))]
inst_ix = content[0] if content else max(1, C["n_text"] - 2)
center = 8 * 16 + 8

plan_a_vecs = [
    ("center_patch L16",  C["H_patch"][16][center]),
    (f"instr {C['text_tokens'][inst_ix][1]!r} L16", C["H_text"][16][inst_ix]),
    ("decision L16",      C["H_decision"][16]),
]

print(f"\nA1 strict  max_new_tokens=7  do_sample=False  inject k={INJECT_LAYER}")
print("=" * 78)
for name, vec in plan_a_vecs:
    texts, ids = interpret(
        llm, vla_tok, INTERP_IDS, SLOTS, vec,
        max_new_tokens=7, do_sample=False, return_ids=True,
    )
    raw_ids = ids[0]
    print(f"\n{name}")
    print(f"  generated ids: {raw_ids}")
    print(f"  decoded text : {texts[0]!r}")
    if len(raw_ids) >= 7:
        acts, bins = tokens_to_actions(vla, raw_ids[:7])
        print(f"  bins         : {bins.tolist()}")
        print(f"  actions      : {fmt_action(acts)}")
        print(f"  Δ vs cached  : {fmt_action(acts - baseline)}")
    else:
        print("  (fewer than 7 tokens — not an action 7-tuple)")


## 6.2 A2 — relaxed generation, banned vs unbanned

Same three vectors, `max_new_tokens=24`. Print both the action-token-banned completion (Phase 3 gating, now on injected states) and the unbanned one.

If banned output is coherent English that tracks the vector, Plan A text works despite action SFT. If both are action-token junk or generic, Plan B remains the interpreter.


In [ ]:
print(f"A2 relaxed  max_new_tokens=24  inject k={INJECT_LAYER}")
print("=" * 78)
for name, vec in plan_a_vecs:
    print(f"\n{name}")
    txt_raw = interpret(
        llm, vla_tok, INTERP_IDS, SLOTS, vec,
        max_new_tokens=24, do_sample=False,
    )[0]
    txt_ban = interpret(
        llm, vla_tok, INTERP_IDS, SLOTS, vec,
        max_new_tokens=24, do_sample=False,
        logits_processor=BAN,
    )[0]
    print(f"  UNBANNED : {txt_raw!r}")
    print(f"  BANNED   : {txt_ban!r}")


---
# Done

You now have, in order:

1. A verified SelfIE harness on Llama-2-7b-chat (Phase 1).
2. A working OpenVLA 7-DoF forward pass and token layout (Phase 2).
3. A gating result on whether the OpenVLA LM head still speaks English (Phase 3).
4. Cached patch vs instruction vs decision vs action hidden states (Phase 4).
5. Plan B interpretations of those states through vanilla Llama (Phase 5).
6. Plan A: the same states injected back into OpenVLA, as actions and as (possibly) text (Phase 6).

## Follow-ups (not in this notebook)

- Instruction-swap comparison (section 5.7 recipe)
- Logit-lens baseline (~15 lines, no injection — catches a broken hook)
- Per-layer linear map `W` on the drift probe, if cosine was low
- Causal patching: splice a patch state from image A into image B, check the action
- HTML report of the grids
